In [1]:
import os
import time
import math
import random
import shutil
import h5py
import csv
import json
import gc
import ctypes
import psutil
import numpy as np
import warnings

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.amp import autocast, GradScaler
from sklearn.metrics import classification_report, hamming_loss, accuracy_score

warnings.filterwarnings('ignore')

try:
    libc = ctypes.CDLL('libc.so.6')
    libc.mallopt(-3, 65536)
    libc.mallopt(-4, 65536)
except Exception:
    pass

RUN_MODE = 'FULL_TRAIN'
GLOBAL_START_TIME = time.time()
MAX_RUN_TIME_SEC = 9.0 * 3600
SEED = 9999
RAM_SAFETY_LIMIT = 90.0
MAX_EPOCHS_THIS_RUN = 8

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

def trim_memory():
    try:
        libc.malloc_trim(0)
    except Exception:
        pass

def get_ram_usage():
    return psutil.virtual_memory().percent

WINDOW_SIZE = 2048
MAX_CHUNKS_LIMIT = 50
CHUNK_BUCKETS = (4, 8, 16, 24, 32, 50)
MAX_BATCH_SIZE_TRAIN = 32
MAX_CHUNKS_PER_BATCH_TRAIN = 128
MAX_BATCH_SIZE_EVAL = 64
MAX_CHUNKS_PER_BATCH_EVAL = 256
ACCUM_STEPS = 4
MAX_EPOCHS = 35
PATIENCE = 7
MAX_LR = 1e-4
SANITY_LR = 5e-4
NUM_TRAIN_PARTS = 8
NUM_WORKERS = 0
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

TRAIN_BUFFER_X = np.full((MAX_BATCH_SIZE_TRAIN, MAX_CHUNKS_LIMIT, 4, WINDOW_SIZE), 4, dtype=np.uint8)
TRAIN_BUFFER_Y = np.empty((MAX_BATCH_SIZE_TRAIN, 4), dtype=np.float32)

EVAL_BUFFER_X = np.full((MAX_BATCH_SIZE_EVAL, MAX_CHUNKS_LIMIT, 4, WINDOW_SIZE), 4, dtype=np.uint8)
EVAL_BUFFER_Y = np.empty((MAX_BATCH_SIZE_EVAL, 4), dtype=np.float32)

BASE_H5_PATH = "/kaggle/input/datasets/huykaizer243/evonaps-numeric/IQNet_Numeric_H5"
if not os.path.exists(BASE_H5_PATH):
    for root, _, files in os.walk("/kaggle/input"):
        if any(f.endswith(".h5") for f in files):
            BASE_H5_PATH = root
            break

MODEL_INPUT_DIR = "/kaggle/input/models/huykaizer243/partitionnet-v27-exp11-tee-sum-attn/pytorch/default/2"

def resolve_input_file(target_name):
    direct_path = os.path.join(MODEL_INPUT_DIR, target_name)
    if os.path.exists(direct_path):
        return direct_path
    for root, _, files in os.walk("/kaggle/input"):
        if target_name in files:
            return os.path.join(root, target_name)
    return ""

RESUME_CHECKPOINT_PATH = resolve_input_file('pnet_v27_exp11_chkpt.pth')
RESUME_BEST_MODEL_PATH = resolve_input_file('pnet_v27_exp11_best.pth')
RESUME_LOG_PATH = resolve_input_file('training_log_exp11.csv')

SAVE_CHECKPOINT_PATH = '/kaggle/working/pnet_v27_exp11_chkpt.pth'
SAVE_BEST_MODEL_PATH = '/kaggle/working/pnet_v27_exp11_best.pth'
SAVE_LOG_PATH = '/kaggle/working/training_log_exp11.csv'
SAVE_METRICS_PATH = '/kaggle/working/pnet_v27_exp11_results.json'
SAVE_PREDS_PATH = '/kaggle/working/pnet_v27_exp11_preds.npz'

def bipolar_to_topo(bit_arr):
    bit_str = "".join(['1' if x > 0 else '0' for x in bit_arr])
    mapping = {"1100": 1, "0011": 1, "1010": 2, "0101": 2, "1001": 3, "0110": 3}
    return mapping.get(bit_str, -1)

class NumericRAMDataset(Dataset):
    def __init__(self, h5_paths, max_samples=None):
        raw_paths = h5_paths if isinstance(h5_paths, list) else [h5_paths]
        self.h5_paths = [p for p in raw_paths if os.path.exists(p)]
        if len(self.h5_paths) == 0:
            raise FileNotFoundError(f"No valid H5 files found: {raw_paths}")

        self.data = []
        lengths_list = []
        ys_list = []
        loaded = 0

        for p in self.h5_paths:
            with h5py.File(p, 'r') as f:
                n = f['y'].shape[0]
                seq_ds = f['sequences']
                y_ds = f['y'][:]

                for i in range(n):
                    seqs = seq_ds[i]
                    real_len = len(seqs[0])
                    arr = np.empty((4, real_len), dtype=np.uint8)
                    for c in range(4):
                        arr[c, :] = seqs[c]

                    self.data.append(arr)
                    lengths_list.append(real_len)
                    ys_list.append(y_ds[i] * 2.0 - 1.0)
                    loaded += 1
                    if max_samples and loaded >= max_samples:
                        break
            if max_samples and loaded >= max_samples:
                break
            gc.collect()

        self.lengths = np.array(lengths_list, dtype=np.int32)
        self.ys = np.array(ys_list, dtype=np.float32)
        del lengths_list, ys_list
        gc.collect()
        trim_memory()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.ys[idx]

class TrainChunkBasedDynamicSampler(Sampler):
    def __init__(self, dataset, max_chunks, max_batch, shuffle=True):
        self.dataset = dataset
        self.max_chunks = max_chunks
        self.max_batch = max_batch
        self.shuffle = shuffle
        self._build_batches()

    def _build_batches(self):
        indices = np.argsort(self.dataset.lengths)
        self.batches = []
        curr_b, curr_max_c = [], 0

        for idx in indices:
            chunks = min(math.ceil(self.dataset.lengths[idx] / WINDOW_SIZE), MAX_CHUNKS_LIMIT)
            proposed_c = max(curr_max_c, chunks)

            if (len(curr_b) + 1) * proposed_c > self.max_chunks or len(curr_b) >= self.max_batch:
                if curr_b:
                    self.batches.append(curr_b)
                curr_b, curr_max_c = [idx], chunks
            else:
                curr_b.append(idx)
                curr_max_c = proposed_c

        if curr_b:
            self.batches.append(curr_b)

    def __iter__(self):
        if self.shuffle:
            random.shuffle(self.batches)
        return iter(self.batches)

    def __len__(self):
        return len(self.batches)

def get_bucket_chunks(max_c):
    for b in CHUNK_BUCKETS:
        if max_c <= b:
            return b
    return MAX_CHUNKS_LIMIT

def collate_train_chunks(batch):
    batch_size = len(batch)
    batch_max_c = 1
    for x, _ in batch:
        c = min(math.ceil(x.shape[1] / WINDOW_SIZE), MAX_CHUNKS_LIMIT)
        if c > batch_max_c:
            batch_max_c = c

    bucket_c = get_bucket_chunks(batch_max_c)
    TRAIN_BUFFER_X[:batch_size, :bucket_c].fill(4)

    for i, (x, y) in enumerate(batch):
        seq_len = x.shape[1]
        num_chunks = min(math.ceil(seq_len / WINDOW_SIZE), MAX_CHUNKS_LIMIT)

        if seq_len <= WINDOW_SIZE:
            TRAIN_BUFFER_X[i, 0, :, :seq_len] = x
        else:
            for c_idx in range(num_chunks):
                start = c_idx * WINDOW_SIZE
                end = start + WINDOW_SIZE
                if end > seq_len:
                    start = max(0, seq_len - WINDOW_SIZE)
                    end = seq_len
                TRAIN_BUFFER_X[i, c_idx, :, :] = x[:, start:end]
        TRAIN_BUFFER_Y[i] = y

    x_tensor = torch.from_numpy(TRAIN_BUFFER_X[:batch_size, :bucket_c])
    y_tensor = torch.from_numpy(TRAIN_BUFFER_Y[:batch_size])
    return x_tensor, y_tensor

def collate_eval_chunks(batch):
    batch_size = len(batch)
    batch_max_c = 1
    for x, _ in batch:
        c = min(math.ceil(x.shape[1] / WINDOW_SIZE), MAX_CHUNKS_LIMIT)
        if c > batch_max_c:
            batch_max_c = c

    bucket_c = get_bucket_chunks(batch_max_c)
    EVAL_BUFFER_X[:batch_size, :bucket_c].fill(4)

    for i, (x, y) in enumerate(batch):
        seq_len = x.shape[1]
        num_chunks = min(math.ceil(seq_len / WINDOW_SIZE), MAX_CHUNKS_LIMIT)

        if seq_len <= WINDOW_SIZE:
            EVAL_BUFFER_X[i, 0, :, :seq_len] = x
        else:
            for c_idx in range(num_chunks):
                start = c_idx * WINDOW_SIZE
                end = start + WINDOW_SIZE
                if end > seq_len:
                    start = max(0, seq_len - WINDOW_SIZE)
                    end = seq_len
                EVAL_BUFFER_X[i, c_idx, :, :] = x[:, start:end]
        EVAL_BUFFER_Y[i] = y

    x_tensor = torch.from_numpy(EVAL_BUFFER_X[:batch_size, :bucket_c])
    y_tensor = torch.from_numpy(EVAL_BUFFER_Y[:batch_size])
    return x_tensor, y_tensor

class CombinedTaxaFusion(nn.Module):
    def __init__(self, embed_dim=16, out_dim_per_taxa=16, num_heads=2):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim * 2, out_dim_per_taxa),
            nn.SiLU(),
            nn.Linear(out_dim_per_taxa, out_dim_per_taxa)
        )
        self.attn = nn.MultiheadAttention(out_dim_per_taxa, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(out_dim_per_taxa)
        self.alpha = nn.Parameter(torch.tensor(0.1))

    def forward(self, x_emb):
        x_perm = x_emb.permute(0, 2, 1, 3).contiguous()
        b_n, w, t, d = x_perm.shape
        x_sum = x_perm.sum(dim=2, keepdim=True)
        feats = []
        for i in range(4):
            x_i = x_perm[:, :, i, :]
            x_context = x_sum.squeeze(2) - x_i
            combined = torch.cat([x_i, x_context], dim=-1)
            feats.append(self.mlp(combined))

        x_tee = torch.stack(feats, dim=2).view(b_n * w, t, -1)
        attn_out, _ = self.attn(x_tee, x_tee, x_tee)
        x_out = self.norm(x_tee + self.alpha * attn_out)
        return x_out.view(b_n, w, -1)

class DilatedContractingBlock(nn.Module):
    def __init__(self, in_channels=64, mid_channels=64, out_channels=32):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv1d(in_channels, mid_channels, kernel_size=7, padding=3, dilation=1),
            nn.BatchNorm1d(mid_channels),
            nn.SiLU(),
            nn.MaxPool1d(4, 4)
        )
        self.conv2 = nn.Sequential(
            nn.Conv1d(mid_channels, out_channels, kernel_size=5, padding=4, dilation=2),
            nn.BatchNorm1d(out_channels),
            nn.SiLU(),
            nn.MaxPool1d(4, 4)
        )
        self.res_proj = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=16),
            nn.BatchNorm1d(out_channels)
        )

    def forward(self, x):
        res = self.res_proj(x)
        h = self.conv1(x)
        h = self.conv2(h)
        return F.silu(h + res)

class ChunkAttentionAggregation(nn.Module):
    def __init__(self, embed_dim=32):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.Tanh(),
            nn.Linear(embed_dim // 2, 1)
        )

    def forward(self, chunk_feats, chunk_mask):
        scores = self.attn(chunk_feats).squeeze(-1)
        scores = scores.masked_fill(chunk_mask, -1e4)
        weights = F.softmax(scores, dim=1)
        weights = torch.nan_to_num(weights, nan=0.0)
        return (chunk_feats * weights.unsqueeze(-1)).sum(dim=1)

class PartitionNet_V27_Exp11_TEESumAttn(nn.Module):
    def __init__(self, embed_dim=16):
        super().__init__()
        self.embedding = nn.Embedding(5, embed_dim)
        self.tef = CombinedTaxaFusion(embed_dim=embed_dim, out_dim_per_taxa=16, num_heads=2)

        self.backbone = DilatedContractingBlock(in_channels=64, mid_channels=64, out_channels=32)
        self.pos_attn = nn.MultiheadAttention(32, num_heads=4, batch_first=True)
        self.chunk_aggregator = ChunkAttentionAggregation(embed_dim=32)

        self.head = nn.Sequential(
            nn.LayerNorm(32),
            nn.Linear(32, 32),
            nn.SiLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 4)
        )

    def forward(self, x_chunks):
        b, n, t, w = x_chunks.shape
        chunk_mask = (x_chunks == 4).all(dim=-1).all(dim=-1)
        x_local = x_chunks.view(b * n, t, w)
        token_mask = (x_local != 4).any(dim=1).float()

        x_emb = self.embedding(x_local.long()) * token_mask.unsqueeze(1).unsqueeze(-1)
        x_proj = self.tef(x_emb) * token_mask.unsqueeze(-1)

        x_in = x_proj.permute(0, 2, 1)
        h_spatial = self.backbone(x_in)

        x_spatial = h_spatial.permute(0, 2, 1)
        x_attn, _ = self.pos_attn(x_spatial, x_spatial, x_spatial)
        chunk_feats = x_attn.max(dim=1)[0].view(b, n, 32)

        h_seq = self.chunk_aggregator(chunk_feats, chunk_mask)
        logits = self.head(h_seq)
        logits = logits - logits.mean(dim=-1, keepdim=True)
        return torch.clamp(logits, min=-15.0, max=15.0)

class StableSymmetricBipolarLoss(nn.Module):
    def __init__(self, lambda_penalty=0.5, temperature=3.0):
        super().__init__()
        self.lambda_penalty = lambda_penalty
        self.temperature = temperature

    def forward(self, logits, targets):
        logits_f32 = logits.float()
        targets_f32 = targets.float()

        zeros = torch.zeros_like(logits_f32)
        loss_y = torch.logaddexp(zeros, -targets_f32 * logits_f32).mean(dim=1)
        loss_minus_y = torch.logaddexp(zeros, targets_f32 * logits_f32).mean(dim=1)
        symmetric_margin_loss = torch.min(loss_y, loss_minus_y).mean()

        preds_soft = torch.tanh(logits_f32 / self.temperature)
        sum_preds = preds_soft.sum(dim=1) / 4.0
        constraint_penalty = torch.mean(sum_preds ** 2)

        total_loss = symmetric_margin_loss + (self.lambda_penalty * constraint_penalty)
        return total_loss, symmetric_margin_loss.detach(), (self.lambda_penalty * constraint_penalty).detach()

def clean_state_dict(sd):
    cleaned = {}
    for k, v in sd.items():
        if k.startswith('module.'):
            k = k[7:]
        if k.startswith('_orig_mod.'):
            k = k[10:]
        cleaned[k] = v
    return cleaned

def decode_topos(logits):
    top2_idx = np.argsort(-logits, axis=1)[:, :2]
    preds_top2 = np.full_like(logits, -1.0)
    np.put_along_axis(preds_top2, top2_idx, 1.0, axis=1)
    return np.array([bipolar_to_topo(b) for b in preds_top2])

def compute_metrics(logits, labels):
    preds_strict = np.where(logits > 0.0, 1.0, -1.0)
    match_orig = (preds_strict == labels).all(axis=1)
    match_flip = (preds_strict == -labels).all(axis=1)
    exact_match = (match_orig | match_flip).mean()

    preds_aligned = np.where(match_flip[:, None], -preds_strict, preds_strict)
    y_true_bin = (labels > 0).astype(int)
    preds_aligned_bin = (preds_aligned > 0).astype(int)
    ham = hamming_loss(y_true_bin, preds_aligned_bin)

    true_topos = [bipolar_to_topo(b) for b in labels]
    pred_topos = decode_topos(logits)
    topo_acc = accuracy_score(true_topos, pred_topos)
    report = classification_report(true_topos, pred_topos, target_names=['Topo 1', 'Topo 2', 'Topo 3'], digits=4, output_dict=True)

    return exact_match, ham, topo_acc, report

def evaluate(model, loader):
    model.eval()
    total_samples = len(loader.dataset)
    logits_arr = np.empty((total_samples, 4), dtype=np.float32)
    labels_arr = np.empty((total_samples, 4), dtype=np.float32)
    ptr = 0

    with torch.no_grad():
        for inputs, labels in loader:
            bs = labels.size(0)
            inputs = inputs.to(DEVICE, non_blocking=True)
            with autocast('cuda'):
                out = model(inputs)
            logits_arr[ptr : ptr + bs] = out.cpu().float().numpy()
            labels_arr[ptr : ptr + bs] = labels.numpy()
            ptr += bs

    exact_match, ham, topo_acc, report = compute_metrics(logits_arr, labels_arr)
    del logits_arr, labels_arr
    return exact_match, ham, topo_acc, report

#SANITY_CHECK
def run_sanity_check():
    pass

#FULL_TRAIN
def run_train():
    train_files = [f"{BASE_H5_PATH}/iqnet_train_part{i}.h5" for i in range(1, NUM_TRAIN_PARTS + 1)]
    val_file = f"{BASE_H5_PATH}/iqnet_val.h5"
    test_file = f"{BASE_H5_PATH}/iqnet_test.h5"

    t0_load = time.time()
    train_dataset = NumericRAMDataset(train_files)
    train_sampler = TrainChunkBasedDynamicSampler(train_dataset, MAX_CHUNKS_PER_BATCH_TRAIN, MAX_BATCH_SIZE_TRAIN, shuffle=True)
    train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=NUM_WORKERS, pin_memory=False, collate_fn=collate_train_chunks)

    val_dataset = NumericRAMDataset([val_file])
    val_sampler = TrainChunkBasedDynamicSampler(val_dataset, MAX_CHUNKS_PER_BATCH_EVAL, MAX_BATCH_SIZE_EVAL, shuffle=False)
    val_loader = DataLoader(val_dataset, batch_sampler=val_sampler, num_workers=NUM_WORKERS, pin_memory=False, collate_fn=collate_eval_chunks)

    print(f"Loaded Train: {len(train_dataset)} | Val: {len(val_dataset)} in {time.time() - t0_load:.1f}s | Base RAM: {get_ram_usage():.1f}%", flush=True)

    model = PartitionNet_V27_Exp11_TEESumAttn().to(DEVICE)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)

    criterion = StableSymmetricBipolarLoss(lambda_penalty=0.5, temperature=3.0)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=MAX_LR, weight_decay=1e-3)

    steps_per_epoch = math.ceil(len(train_loader) / ACCUM_STEPS)
    total_steps = steps_per_epoch * MAX_EPOCHS
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=MAX_LR, total_steps=total_steps, pct_start=0.1)
    scaler = GradScaler('cuda')

    START_EPOCH = 1
    best_val_acc = 0.0
    best_epoch = 1
    epochs_no_improve = 0

    if RESUME_CHECKPOINT_PATH and os.path.exists(RESUME_CHECKPOINT_PATH):
        checkpoint = torch.load(RESUME_CHECKPOINT_PATH, map_location=DEVICE)
        model_core = model.module if isinstance(model, nn.DataParallel) else model
        model_core.load_state_dict(clean_state_dict(checkpoint['model_state']))
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        START_EPOCH = checkpoint['epoch'] + 1
        best_val_acc = checkpoint.get('best_val_acc', 0.0)
        best_epoch = checkpoint.get('best_epoch', START_EPOCH - 1)
        epochs_no_improve = checkpoint.get('epochs_no_improve', 0)
        print(f"Resumed from {RESUME_CHECKPOINT_PATH}", flush=True)
        print(f"Start Epoch: {START_EPOCH} | Best Acc: {best_val_acc*100:.2f}% (Ep {best_epoch}) | Patience Counter: {epochs_no_improve}/{PATIENCE}", flush=True)
    else:
        print("No resume checkpoint found. Starting from Epoch 1.", flush=True)

    target_end_epoch = min(START_EPOCH + MAX_EPOCHS_THIS_RUN - 1, MAX_EPOCHS)
    print(f"Planned Stage Range: Epoch {START_EPOCH} -> Epoch {target_end_epoch} (Total target: {MAX_EPOCHS})", flush=True)

    if RESUME_BEST_MODEL_PATH and os.path.exists(RESUME_BEST_MODEL_PATH) and not os.path.exists(SAVE_BEST_MODEL_PATH):
        shutil.copy(RESUME_BEST_MODEL_PATH, SAVE_BEST_MODEL_PATH)

    if RESUME_LOG_PATH and os.path.exists(RESUME_LOG_PATH) and not os.path.exists(SAVE_LOG_PATH):
        shutil.copy(RESUME_LOG_PATH, SAVE_LOG_PATH)

    write_header = not os.path.exists(SAVE_LOG_PATH)
    with open(SAVE_LOG_PATH, mode='a' if not write_header else 'w', newline='') as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow(['Epoch', 'Time', 'LR', 'Alpha', 'RAM_Pct', 'Tr_Loss', 'Tr_Margin', 'Tr_Penalty', 'Va_Exact', 'Va_Ham', 'Va_TopoAcc'])
            f.flush()
            os.fsync(f.fileno())

    emergency_exit = False

    for ep in range(START_EPOCH, target_end_epoch + 1):
        t0 = time.time()
        model.train()
        optimizer.zero_grad(set_to_none=True)

        tr_loss_sum = 0.0
        tr_margin_sum = 0.0
        tr_penalty_sum = 0.0
        tr_total = 0
        current_lr = scheduler.get_last_lr()[0]

        for idx, (inputs, labels) in enumerate(train_loader):
            inputs = inputs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with autocast('cuda'):
                out = model(inputs)
                loss_tuple = criterion(out, labels)
                loss = loss_tuple[0]
                margin_loss = loss_tuple[1]
                penalty_loss = loss_tuple[2]

            scaler.scale(loss / ACCUM_STEPS).backward()

            if (idx + 1) % ACCUM_STEPS == 0 or (idx + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()

                if scheduler.last_epoch < scheduler.total_steps:
                    scheduler.step()

                optimizer.zero_grad(set_to_none=True)

            bs = labels.size(0)
            tr_loss_sum += loss.item() * bs
            tr_margin_sum += margin_loss.item() * bs
            tr_penalty_sum += penalty_loss.item() * bs
            tr_total += bs

            if (idx + 1) % 500 == 0:
                trim_memory()
                current_ram = get_ram_usage()
                if current_ram >= RAM_SAFETY_LIMIT:
                    print(f"\nIn-Batch Circuit Breaker hit at batch {idx+1}: RAM {current_ram:.1f}% >= {RAM_SAFETY_LIMIT}%. Halting stage cleanly.", flush=True)
                    emergency_exit = True
                    break

        if emergency_exit:
            break

        tr_loss_avg = tr_loss_sum / tr_total if tr_total > 0 else 0.0
        tr_margin_avg = tr_margin_sum / tr_total if tr_total > 0 else 0.0
        tr_penalty_avg = tr_penalty_sum / tr_total if tr_total > 0 else 0.0

        va_em, va_ham, va_acc, _ = evaluate(model, val_loader)

        elapsed = time.time() - t0
        curr_ram = get_ram_usage()

        model_core = model.module if isinstance(model, nn.DataParallel) else model
        current_alpha = model_core.tef.alpha.item()

        print(f"Ep {ep:2d}/{MAX_EPOCHS} | {elapsed:.1f}s | LR: {current_lr:.2e} | Alpha: {current_alpha:.4f} | RAM: {curr_ram:.1f}% | Tr_Loss: {tr_loss_avg:.4f} | Va_Exact: {va_em:.4f} | Va_Ham: {va_ham:.4f} | Va_TopoAcc: {va_acc*100:.2f}%", flush=True)

        with open(SAVE_LOG_PATH, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([ep, round(elapsed, 1), f"{current_lr:.2e}", round(current_alpha, 4), round(curr_ram, 1), round(tr_loss_avg, 4), round(tr_margin_avg, 4), round(tr_penalty_avg, 4), round(va_em, 4), round(va_ham, 4), round(va_acc, 4)])
            f.flush()
            os.fsync(f.fileno())

        state_dict = model_core._orig_mod.state_dict() if hasattr(model_core, '_orig_mod') else model_core.state_dict()

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_epoch = ep
            epochs_no_improve = 0
            torch.save(state_dict, SAVE_BEST_MODEL_PATH)
        else:
            epochs_no_improve += 1

        torch.save({
            'epoch': ep,
            'model_state': state_dict,
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'best_val_acc': best_val_acc,
            'best_epoch': best_epoch,
            'epochs_no_improve': epochs_no_improve
        }, SAVE_CHECKPOINT_PATH)

        trim_memory()
        gc.collect()
        torch.cuda.empty_cache()

        if curr_ram >= RAM_SAFETY_LIMIT:
            print(f"\nEnd-of-Epoch Circuit Breaker hit: RAM {curr_ram:.1f}% >= {RAM_SAFETY_LIMIT}%. Halting stage cleanly.", flush=True)
            break

        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping triggered at epoch {ep} (No improvement for {PATIENCE} epochs)", flush=True)
            break

        if (time.time() - GLOBAL_START_TIME) > MAX_RUN_TIME_SEC:
            print("Session time limit reached. Halting stage.", flush=True)
            break

    del train_dataset, train_sampler, train_loader, val_dataset, val_sampler, val_loader
    gc.collect()
    torch.cuda.empty_cache()
    trim_memory()

    test_dataset = NumericRAMDataset([test_file])
    test_sampler = TrainChunkBasedDynamicSampler(test_dataset, MAX_CHUNKS_PER_BATCH_EVAL, MAX_BATCH_SIZE_EVAL, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_sampler=test_sampler, num_workers=NUM_WORKERS, pin_memory=False, collate_fn=collate_eval_chunks)

    eval_weights = SAVE_BEST_MODEL_PATH if os.path.exists(SAVE_BEST_MODEL_PATH) else SAVE_CHECKPOINT_PATH
    eval_model = PartitionNet_V27_Exp11_TEESumAttn().to(DEVICE)
    eval_state = torch.load(eval_weights, map_location=DEVICE)
    if isinstance(eval_state, dict) and 'model_state' in eval_state:
        eval_state = eval_state['model_state']
    eval_model.load_state_dict(clean_state_dict(eval_state))
    eval_model.eval()

    t0_eval = time.time()
    test_total = len(test_dataset)
    test_logits = np.empty((test_total, 4), dtype=np.float32)
    test_labels_all = np.empty((test_total, 4), dtype=np.float32)
    ptr = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            bs = labels.size(0)
            inputs = inputs.to(DEVICE, non_blocking=True)
            with autocast('cuda'):
                out = eval_model(inputs)
            test_logits[ptr : ptr + bs] = out.cpu().float().numpy()
            test_labels_all[ptr : ptr + bs] = labels.numpy()
            ptr += bs

    te_em, te_ham, te_acc, te_rep = compute_metrics(test_logits, test_labels_all)
    print(f"Stage evaluation finished in {time.time() - t0_eval:.1f}s", flush=True)

    np.savez_compressed(SAVE_PREDS_PATH, logits=test_logits, labels=test_labels_all)

    print("\n" + "=" * 80)
    print("EXP11 (TEE-SUM + TEE-ATTN) STAGE TEST SUMMARY (CANONICAL PASS 0)")
    print("=" * 80)
    print(f"Exact Match:     {te_em:.4f}")
    print(f"Hamming Loss:    {te_ham:.4f}")
    print(f"Topo Accuracy:   {te_acc*100:.2f}%")
    print(f"Delta vs Anchor: {(te_acc - 0.8204)*100:+.2f}%")
    print("=" * 80)

    results_export = {
        "experiment": "pnet-v27-exp11-tee-sum-attn",
        "stage_range": f"Epoch {START_EPOCH} -> {target_end_epoch}",
        "pass0_exact_match": float(te_em),
        "pass0_hamming_loss": float(te_ham),
        "pass0_topo_acc": float(te_acc),
        "delta_vs_v27_anchor": float((te_acc - 0.8204) * 100),
        "best_epoch": int(best_epoch),
        "best_val_acc": float(best_val_acc),
        "classification_report": te_rep
    }

    with open(SAVE_METRICS_PATH, 'w') as f_json:
        json.dump(results_export, f_json, indent=4)

    del test_dataset, test_sampler, test_loader, eval_model, test_logits, test_labels_all
    gc.collect()
    trim_memory()
    torch.cuda.empty_cache()
    print("Cell execution completed cleanly.", flush=True)

def main():
    if not RUN_MODE:
        return
    print(f"Execution Mode: {RUN_MODE}", flush=True)
    if RUN_MODE == 'SANITY_CHECK':
        run_sanity_check()
    elif RUN_MODE == 'FULL_TRAIN':
        run_train()

if __name__ == "__main__":
    main()

Execution Mode: FULL_TRAIN
Loaded Train: 415303 | Val: 51401 in 131.7s | Base RAM: 15.9%
Resumed from /kaggle/input/models/huykaizer243/partitionnet-v27-exp11-tee-sum-attn/pytorch/default/2/pnet_v27_exp11_chkpt.pth
Start Epoch: 21 | Best Acc: 81.28% (Ep 20) | Patience Counter: 0/7
Planned Stage Range: Epoch 21 -> Epoch 28 (Total target: 35)
Ep 21/35 | 1860.1s | LR: 4.63e-05 | Alpha: 0.2055 | RAM: 28.1% | Tr_Loss: 0.2022 | Va_Exact: 0.7562 | Va_Ham: 0.1530 | Va_TopoAcc: 81.09%
Ep 22/35 | 1858.1s | LR: 4.13e-05 | Alpha: 0.2061 | RAM: 35.8% | Tr_Loss: 0.2013 | Va_Exact: 0.7573 | Va_Ham: 0.1523 | Va_TopoAcc: 81.23%
Ep 23/35 | 1858.0s | LR: 3.65e-05 | Alpha: 0.2057 | RAM: 43.6% | Tr_Loss: 0.2006 | Va_Exact: 0.7530 | Va_Ham: 0.1565 | Va_TopoAcc: 81.18%
Ep 24/35 | 1861.5s | LR: 3.17e-05 | Alpha: 0.2066 | RAM: 51.3% | Tr_Loss: 0.2000 | Va_Exact: 0.7549 | Va_Ham: 0.1543 | Va_TopoAcc: 81.10%
Ep 25/35 | 1854.2s | LR: 2.72e-05 | Alpha: 0.2073 | RAM: 59.0% | Tr_Loss: 0.1994 | Va_Exact: 0.7549 | Va_